In [4]:
import pandas as pd
import uuid
import json
import os
from pathlib import Path
from math import floor

# Dossier contenant les fichiers CSV
input_folder = "Adjudication"
output_folder = "JSON entraînementY"
Path(output_folder).mkdir(parents=True, exist_ok=True)

# Fonction de conversion CSV → JSON (avec retour des annotations séparées)
def convert_csv_to_json(file_path):
    try:
        df = pd.read_csv(file_path, sep=';', encoding="utf-8")
        tokens = df["Token"].fillna("").astype(str).tolist()
        full_text = " ".join(tokens).replace("\n", "").strip()

        annotations = []
        for i, row in df.iterrows():
            token = str(row["Token"])
            label = row["Label_Maj"]
            quaero_label = row["Quaero_Maj"] if pd.notna(row["Quaero_Maj"]) else None

            if pd.notna(label):
                start = full_text.find(token)
                end = start + len(token)

                tag_name = f"{label}, {quaero_label}" if label == "LOC" and quaero_label else label

                annotations.append({
                    "id": str(uuid.uuid4()),
                    "tag_id": str(uuid.uuid4()),
                    "start": start,
                    "end": end,
                    "example_id": str(uuid.uuid4()),
                    "tag_name": tag_name,
                    "value": token,
                    "correct": None,
                    "human_annotations": [],
                    "model_annotations": []
                })

        example = {
            "id": str(uuid.uuid4()),
            "content": full_text,
            "metadata": {},
            "annotations": annotations
        }

        print(f"📂 Fichier : {file_path}")
        print(f"   🔹 Tokens : {len(tokens)}")
        print(f"   🔹 Annotations : {len(annotations)}")

        return example
    except Exception as e:
        print(f"❌ Erreur avec {file_path} : {e}")
        return None

# Chargement de tous les fichiers
examples_all = []
examples_ref = []
examples_kraken = []
examples_tesseract = []
files_processed = []

for file_name in os.listdir(input_folder):
    if file_name.endswith("consolide.csv"):
        path = os.path.join(input_folder, file_name)
        example = convert_csv_to_json(path)
        if example:
            examples_all.append(example)
            files_processed.append(file_name)
            if "REF" in file_name:
                examples_ref.append(example)
            if "Kraken" in file_name:
                examples_kraken.append(example)
            if "Tesseract" in file_name:
                examples_tesseract.append(example)

# Sauvegarde des fichiers JSON
def save_json(data, name):
    out_path = os.path.join(output_folder, name)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump({"examples": data}, f, indent=4, ensure_ascii=False)
    print(f"✅ Fichier sauvegardé : {out_path} ({len(data)} exemples)")

# Fichier global
save_json(examples_all, "ensemble_complet.json")

# Fichiers par système
save_json(examples_ref, "ensemble_REF.json")
save_json(examples_kraken, "ensemble_Kraken.json")
save_json(examples_tesseract, "ensemble_Tesseract.json")

# Génération des fichiers à 25%, 50%, 75%
for percent in [25, 50, 75]:
    print(f"\n📊 Génération du sous-ensemble à {percent}%")
    subset = []
    total_annots = 0

    for ex in examples_all:
        n = len(ex["annotations"])
        n_selected = floor(n * percent / 100)
        subset.append({
            **ex,
            "annotations": ex["annotations"][:n_selected]
        })
        print(f"   📄 {ex['id'][:8]}... → {n_selected} annotations ({percent}%)")
        total_annots += n_selected

    save_json(subset, f"ensemble_{percent}pourcent.json")
    print(f"🔢 Total annotations conservées : {total_annots}")

print("\n🏁 Traitement terminé.")

📂 Fichier : Adjudication/adjudication_CARRAUD_petite-Jeanne_TesseractFra-PNG_spaCy3.7.2-lg_consolide.csv
   🔹 Tokens : 6500
   🔹 Annotations : 380
📂 Fichier : Adjudication/adjudication_CARRAUD_petite-Jeanne_REF_spaCy3.7.2-lg_consolide.csv
   🔹 Tokens : 6500
   🔹 Annotations : 383
📂 Fichier : Adjudication/adjudication_CARRAUD_petite-Jeanne_Kraken-base_spaCy3.7.2-lg 1_consolide.csv
   🔹 Tokens : 6500
   🔹 Annotations : 389
📂 Fichier : Adjudication/adjudication_DAUDET_petit-chose_Kraken-base_spaCy3.7.2-lg_consolide.csv
   🔹 Tokens : 6500
   🔹 Annotations : 270
📂 Fichier : Adjudication/adjudication_DAUDET_petit-chose_REF_spaCy3.7.2-lg_consolide.csv
   🔹 Tokens : 6500
   🔹 Annotations : 280
📂 Fichier : Adjudication/adjudication_DAUDET_petit-chose_TesseractFra-PNG_spaCy3.7.2-lg_consolide.csv
   🔹 Tokens : 6500
   🔹 Annotations : 250
✅ Fichier sauvegardé : JSON entraînementY/ensemble_complet.json (6 exemples)
✅ Fichier sauvegardé : JSON entraînementY/ensemble_REF.json (2 exemples)
✅ Fichier s